# TTS bootcamp – validation experiments for Task 3 and Task 4 (Colab, T4 GPU)

Same setup as `run_all.ipynb` (HF_TOKEN secret, T4, restart once after install).
Runs only the 24 **validation** items for each config and prints one comparison table.
Budget: about 1.5 GPU hours for both phases; set `PHASES` to run one at a time.
At the end, download `exp_results.zip` and upload it to Claude.

In [ ]:
ROLL = "ROLLNO"                       # your roll number
REPO = "/content/InterIITNLP2"
BRANCH = "claude/cool-tesla-oqlizq"
INDICF5_CODE = "13f7c4d627cc10111aea8fe9c0039462cacacdc7"   # github.com/ai4bharat/IndicF5 commit (code)
# The model weights stay pinned to the Hugging Face revision in kit/baseline/config.yaml

import os, subprocess, sys
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-6000:])
        if check:
            raise RuntimeError(cmd)
    return r

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

if not os.path.isdir(f"{REPO}/submission"):
    tok = secret("GH_TOKEN")
    url = f"https://{tok}@github.com/Mihir269/InterIITNLP2.git" if tok else "https://github.com/Mihir269/InterIITNLP2.git"
    sh(f"git clone --depth 1 -b {BRANCH} {url} {REPO}")
else:
    sh(f"git -C {REPO} pull --ff-only", check=False)
os.chdir(f"{REPO}/submission")
print(os.getcwd(), os.listdir("."))

In [ ]:
# Install (keeps Colab's own torch). Skipped on the second run after a restart.
# IndicF5's setup.py asks for numpy<=1.26.4, but the kit needs numpy>=2.1 (and it works with it),
# so IndicF5 is installed with --no-deps and its dependencies are installed separately.
B = f"{REPO}/kit/baseline"
try:
    import f5_tts, speechbrain  # noqa: F401
    print("already installed")
except ImportError:
    C = f"-c {B}/constraints_colab.txt"
    sh(f"pip install -q -r {B}/requirements_colab.txt {C}")
    sh(f"pip install -q --no-deps git+https://github.com/ai4bharat/IndicF5.git@{INDICF5_CODE}")
    deps = ('accelerate cached_path click datasets "ema_pytorch>=0.5.2" "hydra-core>=1.3.0" jieba librosa '
            'matplotlib pydub pypinyin safetensors soundfile tomli torchdiffeq "tqdm>=4.65.0" '
            'transformers_stream_generator vocos wandb "x_transformers>=1.31.14" '
            '"audioop-lts; python_version>=\'3.13\'"')
    sh(f"pip install -q {deps} {C}")
    sh("pip check", check=False)   # a numpy<=1.26.4 complaint from f5-tts is expected and harmless
    print(">>> Installed. The runtime restarts itself now; when it reconnects, do Runtime > Run all again. <<<")
    os.kill(os.getpid(), 9)   # Colab restarts the kernel; the next Run all skips this install

In [ ]:
from huggingface_hub import login
login(token=secret("HF_TOKEN"))
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU!")

In [ ]:
# Load the model once; applies the kit's length fix (may ask for one restart)
sys.path.insert(0, f"{REPO}/kit")
from baseline import patch_length
print("length fix:", patch_length.ensure_patched() or "already patched")
from baseline.tts import get_engine
try:
    get_engine()
    print("IndicF5 loaded OK")
except RuntimeError as e:
    print(e)
    raise SystemExit(">>> Runtime > Restart session, then Run all again. <<<")

In [ ]:
# Round 1 (done): "t3 t4".  Round 2: "t3b t4b" = syllable length only for sentences with English
# words, a seed-1/seed-2 check on the Hinglish items, and 10/12 steps on top. base + idea re-run first
# (same-session reference for timing), ~45 min in total.
# Round 3: "t2c" = spelled abbreviations with commas (ए, टी, एम) on the 8 hard items, seeds 0-2 (~15 min)
# Round 4: "cfg" = cfg_strength 1.5 / 2.5 / 3.0 on the final Task 3 system (24 items, seed 0, ~35 min).
#          Then, only if one beats 2.0 (CER 0.0532): PHASES = "cfg_seeds" with CFGBEST set to it (~45 min).
PHASES = "cfg"
CFGBEST = "2.5"
r = subprocess.run(f'KIT={REPO}/kit PHASES="{PHASES}" CFGBEST={CFGBEST} bash code/run_experiments.sh 2>&1 | tee -a /content/exp.log',
                   shell=True, text=True)
print(open("/content/exp.log").read()[-5000:])

In [ ]:
sh("python code/analysis/compare.py --res code/results/exp --ref cfg20 --base base")


In [ ]:
sh(f"cd {REPO}/submission && zip -qr /content/exp_results.zip code/results/exp /content/exp.log", check=False)
from google.colab import files
files.download("/content/exp_results.zip")